In [88]:
import pandas as pd
from github_helper import from_github
import ast

# This notebook merges votes, voting_sessions and actor information

In [89]:
df_votes = pd.read_csv(from_github("/voting-data/df_votes_all_periods.csv"))
df_voting_sessions = pd.read_csv(from_github("/voting-data/voting_sessions_enriched.csv"))
party_per_period = pd.read_csv(from_github("/actor-data/party_per_period.csv"))

In [90]:
# df_votes.head(2)

In [91]:
relevant_topics_for_joining_on_votes = ["afstemning_id", "afstemning_nummer", "afstemning_vedtaget", "Period", "primary_topic", "all_topics", "møde_dato", 'møde_year_month'] #The last one is a homemade one ya know
df_voting_sessions['møde_dato'] = pd.to_datetime(df_voting_sessions['møde_dato'])
df_voting_sessions['møde_year_month'] = df_voting_sessions['møde_dato'].dt.to_period('M')
# df_voting_sessions.head()

In [92]:
# party_per_period.head()

In [93]:
votes_enriched = df_votes.merge(df_voting_sessions[relevant_topics_for_joining_on_votes], how = "left", on = "afstemning_id")
# votes_enriched.head()

In [94]:
votes_with_party = votes_enriched.merge(party_per_period, how = "left", on = ["aktørid", "Period"])
# votes_with_party.head()

In [95]:
columns_to_drop = ["vote_opdateringsdato"]
df_limited = votes_with_party.drop(columns = columns_to_drop)
df_renamed = df_limited.rename(columns = {"aktør" : "politician"})

print(df_renamed.groupby("Period")["afstemning_id"].nunique())
print(df_renamed.groupby("Period")["aktørid"].nunique())
#Sanity check

Period
65     294
66    1269
67    1778
68    1727
69    2017
70    1688
71    1306
Name: afstemning_id, dtype: int64
Period
65    189
66    216
67    241
68    228
69    237
70    219
71    235
Name: aktørid, dtype: int64


In [96]:
for period in [65, 66, 67, 68, 69, 70, 71]:
    votes_p = df_renamed[df_renamed["Period"] == period]
    votes_p.to_csv(f"./voting-data/votes_enriched_p{period}.csv", index = False)